# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullahhashmi01/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from pathlib import Path
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded:", df.shape)

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Dataset loaded: (30000, 44)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
required_columns = [
    "content_id",
    "impressions_90d", # Changed from "impressions" to "impressions_90d"
    "ctr",
    "avg_position"
]

# --- Added for debugging ---
print("DataFrame columns:", df.columns.tolist())
# --- End debugging ---

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing_columns)
    )

work = df[required_columns].copy()

# Convert fields to numeric
for column in [
    "impressions_90d", # Changed from "impressions" to "impressions_90d"
    "ctr",
    "avg_position"
]:
    work[column] = pd.to_numeric(
        work[column],
        errors="coerce"
    )

# Zero means no position data
work["avg_position"] = (
    work["avg_position"].replace(0, np.nan)
)

impression_cutoff = work["impressions_90d"].quantile(0.75) # Changed from "impressions" to "impressions_90d"
ctr_cutoff = work["ctr"].quantile(0.25)

print("High-impression cutoff:", round(impression_cutoff, 2))
print("Low-CTR cutoff:", round(ctr_cutoff, 2))
print(
    "Missing position records:",
    work["avg_position"].isna().sum()
)

DataFrame columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
High-impression cutoff: 3615.25
Low-CTR cutoff: 0.0
Missing position records: 1205


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# High impressions = high opportunity
work["impression_score"] = (
    work["impressions_90d"] # Changed from "impressions" to "impressions_90d"
    .fillna(0)
    .rank(pct=True)
)

# Low CTR = high opportunity
ctr_filled = work["ctr"].fillna(
    work["ctr"].median()
)

work["low_ctr_score"] = (
    1 - ctr_filled.rank(pct=True)
)

# Position opportunity
work["position_score"] = 0.0

work.loc[
    work["avg_position"].between(4, 20),
    "position_score"
] = 1.0

work.loc[
    work["avg_position"].between(21, 40),
    "position_score"
] = 0.5

# Final score from 0 to 100
work["action_score"] = 100 * (
    0.50 * work["impression_score"]
    + 0.30 * work["low_ctr_score"]
    + 0.20 * work["position_score"]
)

work["action_score"] = (
    work["action_score"].round(2)
)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def make_reason_codes(row):
    reasons = []

    if row["impressions_90d"] >= impression_cutoff: # Changed from "impressions" to "impressions_90d"
        reasons.append("HIGH_IMPRESSIONS")

    if row["ctr"] <= ctr_cutoff:
        reasons.append("LOW_CTR")

    if pd.isna(row["avg_position"]):
        reasons.append("MISSING_POSITION")

    elif 4 <= row["avg_position"] <= 20:
        reasons.append("POSITION_4_TO_20")

    elif 21 <= row["avg_position"] <= 40:
        reasons.append("POSITION_21_TO_40")

    if not reasons:
        reasons.append("NO_STRONG_RULE_SIGNAL")

    return " | ".join(reasons)

work["reason_code"] = work.apply(
    make_reason_codes,
    axis=1
)

In [10]:
ranked_queue = (
    work.sort_values(
        ["action_score", "impressions_90d"], # Changed 'impressions' to 'impressions_90d'
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = (
    np.arange(1, len(ranked_queue) + 1)
)

output_columns = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "impressions_90d", # Changed 'impressions' to 'impressions_90d'
    "ctr",
    "avg_position"
]

ranked_queue = ranked_queue[output_columns]

output_directory = Path("work/outputs")
output_directory.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_directory
    / "baseline_action_score.csv"
)

ranked_queue.to_csv(
    output_path,
    index=False
)

print("Ranked records:", len(ranked_queue))
print("Output saved to:", output_path)

Ranked records: 30000
Output saved to: work/outputs/baseline_action_score.csv


In [11]:
def recommend_action(reason):
    if "LOW_CTR" in reason:
        return (
            "Review title, description and "
            "search-intent alignment"
        )

    if "POSITION_4_TO_20" in reason:
        return (
            "Review and refresh content for "
            "ranking opportunity"
        )

    if "POSITION_21_TO_40" in reason:
        return (
            "Check content depth and relevance "
            "before prioritising"
        )

    return "Investigate measurements before action"


def confidence_note(reason):
    strong_reasons = [
        "HIGH_IMPRESSIONS",
        "LOW_CTR",
        "POSITION_4_TO_20"
    ]

    reason_count = sum(
        code in reason
        for code in strong_reasons
    )

    if "MISSING_POSITION" in reason:
        return "Low — position data is missing"

    if reason_count >= 3:
        return "Higher — three rule signals agree"

    if reason_count == 2:
        return "Medium — two rule signals agree"

    return "Low — limited rule evidence"


def wrong_if_note(reason):
    if "MISSING_POSITION" in reason:
        return (
            "Position data is unavailable or the "
            "search measurements are incomplete"
        )

    if "LOW_CTR" in reason:
        return (
            "Low CTR is normal for the query mix, "
            "device mix or search intent"
        )

    return (
        "Demand is seasonal, already changing, "
        "or not valuable to the editor"
    )

In [12]:
top_20 = ranked_queue.head(20).copy()

top_20["recommended_action"] = (
    top_20["reason_code"].apply(
        recommend_action
    )
)

top_20["confidence_note"] = (
    top_20["reason_code"].apply(
        confidence_note
    )
)

top_20["what_would_make_it_wrong"] = (
    top_20["reason_code"].apply(
        wrong_if_note
    )
)

review_columns = [
    "rank",
    "content_id",
    "action_score",
    "recommended_action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top_20[review_columns])

,rank,content_id,action_score,recommended_action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_c8e9d6ab9013,93.35,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
1,2,content_f986bd514b6e,90.82,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
2,3,content_ae6d1339904d,89.94,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
3,4,content_825a9788af8d,89.76,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
4,5,content_8ba781dafa55,89.61,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
5,6,content_5d5653c4eb4f,89.37,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
6,7,content_847a841969a2,89.19,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
7,8,content_c82bc0c24241,88.93,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
8,9,content_eb1510f4b5f1,88.43,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
9,10,content_3e79eaafc89d,86.71,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."


In [13]:
top_20 = ranked_queue.head(20).copy()

top_20["recommended_action"] = (
    top_20["reason_code"].apply(
        recommend_action
    )
)

top_20["confidence_note"] = (
    top_20["reason_code"].apply(
        confidence_note
    )
)

top_20["what_would_make_it_wrong"] = (
    top_20["reason_code"].apply(
        wrong_if_note
    )
)

review_columns = [
    "rank",
    "content_id",
    "action_score",
    "recommended_action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top_20[review_columns])

,rank,content_id,action_score,recommended_action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_c8e9d6ab9013,93.35,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
1,2,content_f986bd514b6e,90.82,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
2,3,content_ae6d1339904d,89.94,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
3,4,content_825a9788af8d,89.76,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
4,5,content_8ba781dafa55,89.61,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
5,6,content_5d5653c4eb4f,89.37,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
6,7,content_847a841969a2,89.19,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
7,8,content_c82bc0c24241,88.93,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
8,9,content_eb1510f4b5f1,88.43,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."
9,10,content_3e79eaafc89d,86.71,"Review title, description and search-intent al...",HIGH_IMPRESSIONS | LOW_CTR | POSITION_4_TO_20,Higher — three rule signals agree,"Low CTR is normal for the query mix, device mi..."


In [14]:
def count_strong_reasons(reason):
    strong_codes = [
        "HIGH_IMPRESSIONS",
        "LOW_CTR",
        "POSITION_4_TO_20",
        "POSITION_21_TO_40"
    ]

    return sum(
        code in reason
        for code in strong_codes
    )

top_20["reason_count"] = (
    top_20["reason_code"].apply(
        count_strong_reasons
    )
)

weak_picks = top_20[
    (top_20["reason_count"] <= 1)
    | top_20["reason_code"].str.contains(
        "MISSING_POSITION"
    )
].copy()

print(
    "Weak picks in Top 20:",
    len(weak_picks)
)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "reason_code",
            "confidence_note"
        ]
    ]
)

Weak picks in Top 20: 0


,rank,content_id,action_score,reason_code,confidence_note


In [15]:
score_features = [
    "impressions",
    "ctr",
    "avg_position"
]

blocked_words = [
    "label",
    "target",
    "trend",
    "future",
    "next",
    "product",
    "client_id",
    "query",
    "url"
]

leaked_features = [
    feature for feature in score_features
    if any(
        word in feature.lower()
        for word in blocked_words
    )
]

print("Score features:", score_features)
print("Blocked features found:", leaked_features)

assert leaked_features == []

print("PASS: no obvious label, future-window,")
print("product-flag or private field was used.")

Score features: ['impressions', 'ctr', 'avg_position']
Blocked features found: []
PASS: no obvious label, future-window,
product-flag or private field was used.


In [16]:
assert len(ranked_queue) == len(df)
assert ranked_queue["rank"].is_unique
assert ranked_queue["content_id"].notna().all()
assert ranked_queue["action_score"].between(
    0,
    100
).all()

print("All records were ranked.")
print("Scores are between 0 and 100.")
print("Output file exists:", output_path.exists())

All records were ranked.
Scores are between 0 and 100.
Output file exists: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.